# 04 — Optimización del modelo

## Modelo predictivo para anticipar incumplimientos de SLA en HR Operations  
## (Predictive Model to Anticipate SLA Breaches in HR Operations)

En el notebook anterior se compararon varios modelos de clasificación supervisada.

Aunque el Árbol de Decisión obtuvo el mayor recall para la clase `1`, Random Forest mostró el mejor equilibrio general entre precision, recall y F1-score para detectar incumplimientos de SLA.

Por este motivo, en este notebook se optimizará Random Forest mediante GridSearchCV.

## Objetivo de la optimización

El objetivo de esta fase es mejorar el rendimiento del modelo candidato ajustando sus hiperparámetros.

Un hiperparámetro es una configuración del modelo que no se aprende automáticamente durante el entrenamiento, sino que debe definirse antes.

En este caso, se utilizará `GridSearchCV` para probar diferentes combinaciones de hiperparámetros y seleccionar la mejor según una métrica de evaluación.

La métrica prioritaria será el recall de la clase `1`, porque el objetivo principal del proyecto es detectar tickets que realmente incumplen SLA.

In [1]:
# Importar librerías:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [2]:
ruta_datos = "../data/processed/tickets_hr_sla_model_ready.csv"

df = pd.read_csv(ruta_datos)

df.head()

,Customer Age,Product Purchased,Ticket Type,Ticket Subject,Ticket Priority,Ticket Channel,incumplio_sla
0,48,Dell XPS,Technical issue,Network problem,Low,Social media,0
1,27,Microsoft Office,Billing inquiry,Account access,Low,Social media,0
2,67,Autodesk AutoCAD,Billing inquiry,Data loss,Low,Email,0
3,48,Nintendo Switch,Cancellation request,Data loss,High,Phone,0
4,51,Microsoft Xbox Controller,Product inquiry,Software bug,High,Chat,1


In [3]:
# Ver tamaño del Dataset:
df.shape

(2769, 7)

In [4]:
X = df.drop(columns="incumplio_sla")
y = df["incumplio_sla"]

X.head()

,Customer Age,Product Purchased,Ticket Type,Ticket Subject,Ticket Priority,Ticket Channel
0,48,Dell XPS,Technical issue,Network problem,Low,Social media
1,27,Microsoft Office,Billing inquiry,Account access,Low,Social media
2,67,Autodesk AutoCAD,Billing inquiry,Data loss,Low,Email
3,48,Nintendo Switch,Cancellation request,Data loss,High,Phone
4,51,Microsoft Xbox Controller,Product inquiry,Software bug,High,Chat


In [5]:
y.value_counts(normalize=True).mul(100).round(2)

incumplio_sla
0    61.79
1    38.21
Name: proportion, dtype: float64

## Separación train/test

Se mantiene la misma división utilizada en los notebooks anteriores para que los resultados sean comparables.

Se utiliza:

- 80% de los datos para entrenamiento.
- 20% de los datos para evaluación.
- `stratify=y` para conservar la proporción de clases.
- `random_state=42` para asegurar reproducibilidad.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (2215, 6)
X_test: (554, 6)
y_train: (2215,)
y_test: (554,)


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (2215, 6)
X_test: (554, 6)
y_train: (2215,)
y_test: (554,)


## Preprocesamiento

Se aplicará el mismo preprocesamiento utilizado en la comparación de modelos:

- `StandardScaler` para la variable numérica.
- `OneHotEncoder` para las variables categóricas.

Aunque Random Forest no necesita escalado estrictamente, se mantiene el mismo pipeline para conservar una estructura homogénea y reproducible.

In [8]:
columnas_numericas = ["Customer Age"]

columnas_categoricas = [
    "Product Purchased",
    "Ticket Type",
    "Ticket Subject",
    "Ticket Priority",
    "Ticket Channel"
]

preprocesamiento = ColumnTransformer(
    transformers=[
        ("numericas", StandardScaler(), columnas_numericas),
        ("categoricas", OneHotEncoder(handle_unknown="ignore"), columnas_categoricas)
    ]
)

## Modelo base: Random Forest

Antes de optimizar el modelo, se entrena una versión base de Random Forest.

Esto permitirá comparar el rendimiento antes y después de aplicar GridSearchCV.

In [9]:
modelo_rf_base = Pipeline(
    steps=[
        ("preprocesamiento", preprocesamiento),
        ("modelo", RandomForestClassifier(
            n_estimators=100,
            random_state=42
        ))
    ]
)

modelo_rf_base.fit(X_train, y_train)

y_pred_rf_base = modelo_rf_base.predict(X_test)

In [10]:
accuracy_rf_base = accuracy_score(y_test, y_pred_rf_base)
precision_rf_base = precision_score(y_test, y_pred_rf_base)
recall_rf_base = recall_score(y_test, y_pred_rf_base)
f1_rf_base = f1_score(y_test, y_pred_rf_base)

print("Random Forest base")
print("Accuracy:", round(accuracy_rf_base, 4))
print("Precision clase 1:", round(precision_rf_base, 4))
print("Recall clase 1:", round(recall_rf_base, 4))
print("F1-score clase 1:", round(f1_rf_base, 4))

Random Forest base
Accuracy: 0.7112
Precision clase 1: 0.634
Recall clase 1: 0.5802
F1-score clase 1: 0.6059


## GridSearchCV

GridSearchCV permite probar varias combinaciones de hiperparámetros y seleccionar la que ofrece mejor rendimiento según una métrica definida.

En este caso, se optimizará Random Forest usando como métrica principal el recall, ya que nos interesa detectar la mayor cantidad posible de tickets que incumplen SLA.

In [11]:
pipeline_rf = Pipeline(
    steps=[
        ("preprocesamiento", preprocesamiento),
        ("modelo", RandomForestClassifier(random_state=42))
    ]
)

param_grid = {
    "modelo__n_estimators": [100, 200],
    "modelo__max_depth": [None, 5, 10],
    "modelo__min_samples_split": [2, 5],
    "modelo__min_samples_leaf": [1, 2],
    "modelo__class_weight": [None, "balanced"]
}

In [12]:
pipeline_rf = Pipeline(
    steps=[
        ("preprocesamiento", preprocesamiento),
        ("modelo", RandomForestClassifier(random_state=42))
    ]
)

param_grid = {
    "modelo__n_estimators": [100, 200],
    "modelo__max_depth": [None, 5, 10],
    "modelo__min_samples_split": [2, 5],
    "modelo__min_samples_leaf": [1, 2],
    "modelo__class_weight": [None, "balanced"]
}

## Qué estamos probando

| Hiperparámetro | Qué controla |
|----------------|--------------|
| `n_estimators` | Número de árboles |
| `max_depth` | Profundidad máxima de cada árbol |
| `min_samples_split` | Mínimo de muestras para dividir un nodo |
| `min_samples_leaf` | Mínimo de muestras en una hoja |
| `class_weight` | Penalización para clases desbalanceadas |

In [13]:
grid_search_rf = GridSearchCV(
    estimator=pipeline_rf,
    param_grid=param_grid,
    scoring="recall",
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid_search_rf.fit(X_train, y_train)

Fitting 5 folds for each of 48 candidates, totalling 240 fits


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocesamiento',
                                        ColumnTransformer(transformers=[('numericas',
                                                                         StandardScaler(),
                                                                         ['Customer '
                                                                          'Age']),
                                                                        ('categoricas',
                                                                         OneHotEncoder(handle_unknown='ignore'),
                                                                         ['Product '
                                                                          'Purchased',
                                                                          'Ticket '
                                                                          'Type',
                                                                          'Ticket '
                                                                          'Subject',
                                                                          'Ticket '
                                                                          'Priority',
                                                                          'Ticket '
                                                                          'Channel'])])),
                                       ('modelo',
                                        RandomForestClassifier(random_state=42))]),
             n_jobs=-1,
             param_grid={'modelo__class_weight': [None, 'balanced'],
                         'modelo__max_depth': [None, 5, 10],
                         'modelo__min_samples_leaf': [1, 2],
                         'modelo__min_samples_split': [2, 5],
                         'modelo__n_estimators': [100, 200]},
             scoring='recall', verbose=1)

## Qué significa

| Parámetro | Significado |
|-----------|-------------|
| `scoring="recall"` | Optimiza la detección de incumplimientos |
| `cv=5` | Usa validación cruzada de 5 particiones |
| `n_jobs=-1` | Usa todos los núcleos disponibles |
| `verbose=1` | Muestra progreso |

In [14]:
grid_search_rf.best_params_

{'modelo__class_weight': 'balanced',
 'modelo__max_depth': 5,
 'modelo__min_samples_leaf': 2,
 'modelo__min_samples_split': 5,
 'modelo__n_estimators': 200}

In [15]:
# Ver el mejor recall promedio obtenido en validación cruzada:
grid_search_rf.best_score_

np.float64(0.8155795335885833)

## Evaluación del modelo optimizado

Después de encontrar la mejor combinación de hiperparámetros, se evalúa el modelo optimizado sobre el conjunto de test.

El conjunto de test no se utilizó durante la búsqueda de hiperparámetros, por lo que permite medir el rendimiento del modelo en datos no vistos.

In [16]:
mejor_modelo_rf = grid_search_rf.best_estimator_

y_pred_rf_opt = mejor_modelo_rf.predict(X_test)

In [17]:
accuracy_rf_opt = accuracy_score(y_test, y_pred_rf_opt)
precision_rf_opt = precision_score(y_test, y_pred_rf_opt)
recall_rf_opt = recall_score(y_test, y_pred_rf_opt)
f1_rf_opt = f1_score(y_test, y_pred_rf_opt)

print("Random Forest optimizado")
print("Accuracy:", round(accuracy_rf_opt, 4))
print("Precision clase 1:", round(precision_rf_opt, 4))
print("Recall clase 1:", round(recall_rf_opt, 4))
print("F1-score clase 1:", round(f1_rf_opt, 4))

Random Forest optimizado
Accuracy: 0.7292
Precision clase 1: 0.6115
Recall clase 1: 0.8019
F1-score clase 1: 0.6939


In [18]:
matriz_rf_opt = confusion_matrix(y_test, y_pred_rf_opt)

matriz_rf_opt

array([[234, 108],
       [ 42, 170]])

In [19]:
print(classification_report(y_test, y_pred_rf_opt))

              precision    recall  f1-score   support

           0       0.85      0.68      0.76       342
           1       0.61      0.80      0.69       212

    accuracy                           0.73       554
   macro avg       0.73      0.74      0.73       554
weighted avg       0.76      0.73      0.73       554



In [20]:
comparacion_rf = pd.DataFrame({
    "modelo": ["Random Forest base", "Random Forest optimizado"],
    "accuracy": [accuracy_rf_base, accuracy_rf_opt],
    "precision_clase_1": [precision_rf_base, precision_rf_opt],
    "recall_clase_1": [recall_rf_base, recall_rf_opt],
    "f1_clase_1": [f1_rf_base, f1_rf_opt]
})

comparacion_rf

,modelo,accuracy,precision_clase_1,recall_clase_1,f1_clase_1
0,Random Forest base,0.711191,0.634021,0.580189,0.605911
1,Random Forest optimizado,0.729242,0.611511,0.801887,0.693878


In [21]:
comparacion_rf.round(4)

,modelo,accuracy,precision_clase_1,recall_clase_1,f1_clase_1
0,Random Forest base,0.7112,0.6340,0.5802,0.6059
1,Random Forest optimizado,0.7292,0.6115,0.8019,0.6939


## Interpretación de la optimización

La optimización mediante GridSearchCV permitió comparar diferentes configuraciones de Random Forest.

El objetivo principal fue mejorar el recall de la clase `1`, ya que esta métrica representa la capacidad del modelo para detectar tickets que realmente incumplen SLA.

La comparación entre el modelo base y el modelo optimizado permite evaluar si el ajuste de hiperparámetros mejora la utilidad operativa del modelo.

Si el recall mejora, el modelo optimizado detecta más casos críticos. Si el recall baja pero mejora el F1-score, el modelo puede estar logrando un mejor equilibrio entre detección de incumplimientos y reducción de falsas alarmas.

## Conclusión del notebook

En este notebook se optimizó el modelo Random Forest mediante GridSearchCV.

El objetivo fue mejorar la capacidad del modelo para detectar tickets con riesgo de incumplir SLA.

La selección final del modelo deberá considerar:

- Recall de la clase `1`.
- F1-score de la clase `1`.
- Cantidad de falsos negativos.
- Nivel de falsas alarmas.
- Utilidad del modelo para priorización operativa.

El siguiente paso será interpretar el modelo seleccionado y traducir sus resultados en recomendaciones de negocio.

## Interpretación de la optimización

La optimización mediante GridSearchCV mejoró de forma significativa el rendimiento del modelo Random Forest para el objetivo principal del proyecto: detectar tickets que incumplen SLA.

El modelo base obtuvo un recall de clase `1` de 0,5802, mientras que el modelo optimizado alcanzó un recall de 0,8019.

Esto significa que el modelo optimizado pasó de detectar aproximadamente el 58% de los incumplimientos a detectar aproximadamente el 80% de ellos.

La mejora se logró principalmente mediante el uso de `class_weight="balanced"` y una configuración más controlada de los árboles, con una profundidad máxima de 5 y 200 estimadores.

La matriz de confusión del modelo optimizado muestra:

| Tipo de predicción | Cantidad |
|---|---:|
| Verdaderos negativos | 234 |
| Falsos positivos | 108 |
| Falsos negativos | 42 |
| Verdaderos positivos | 170 |

Desde el punto de vista de negocio, la reducción de falsos negativos es especialmente importante, porque los falsos negativos representan tickets que realmente incumplen SLA pero que el modelo no detecta a tiempo.

Aunque el modelo optimizado genera más falsas alarmas, este trade-off es aceptable en un contexto operativo donde es preferible revisar algunos tickets adicionales antes que dejar sin detectar casos críticos.

## Conclusión del notebook

En este notebook se optimizó el modelo Random Forest mediante GridSearchCV.

La optimización permitió mejorar claramente la capacidad del modelo para detectar incumplimientos de SLA.

El recall de la clase `1` aumentó de 0,5802 a 0,8019, lo que representa una mejora relevante para el objetivo operativo del proyecto.

Aunque la precision de la clase `1` disminuyó ligeramente, el modelo optimizado ofrece mayor utilidad para priorización operativa, ya que reduce de forma importante la cantidad de tickets críticos no detectados.

Por tanto, Random Forest optimizado se selecciona como modelo candidato final del proyecto.

El siguiente paso será interpretar las variables más relevantes del modelo y traducir los resultados en recomendaciones de negocio para HR Operations.